### Import Libraries

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys

import warnings
warnings.filterwarnings("ignore")

### Data Importing

In [23]:
df = pd.read_csv(r"C:\Users\Asus\Documents\Projects\Dhaka-house-price-predictions\notebook\data\dhaka_house_price.csv")
df

,Location,Price,Type,No. Beds,No. Baths,Area,Latitude,Longitude,Region,Sub-region
0,"Sector 10, Uttara, Dhaka",7500000,Apartment,3.0,3.0,1300,23.86846,90.39280,Uttara,Sector 10
1,"Section 11, Mirpur, Dhaka",7280000,Apartment,4.0,4.0,1456,23.81223,90.35967,Mirpur,Section 11
2,"Chowdhuripara, Khilgaon, Dhaka",13000000,Apartment,3.0,3.0,1550,23.75349,90.42469,Khilgaon,Chowdhuripara
3,"Road No 4, Banani, Dhaka",37000000,Apartment,3.0,3.0,2669,23.78855,90.40081,Banani,Road No 4
4,"South Banasree Project, Banasree, Dhaka",3600000,Apartment,2.0,2.0,835,23.76354,90.43180,Banasree,South Banasree Project
...,...,...,...,...,...,...,...,...,...,...
4699,"Middle Monipur, Mirpur, Dhaka",4950000,Apartment,3.0,2.0,1100,23.81223,90.35967,Mirpur,Middle Monipur
4700,"Middle Monipur, Mirpur, Dhaka",4950000,Apartment,3.0,2.0,1100,23.81223,90.35967,Mirpur,Middle Monipur
4701,"Middle Monipur, Mirpur, Dhaka",4950000,Apartment,3.0,2.0,1100,23.81223,90.35967,Mirpur,Middle Monipur
4702,"Middle Monipur, Mirpur, Dhaka",4950000,Apartment,3.0,2.0,1100,23.81223,90.35967,Mirpur,Middle Monipur


In [24]:
print(f"Dataset Shape: {df.shape}")

Dataset Shape: (4704, 10)


### Feature Engineering

In [25]:
# 1. Total Amenities Score (Summing all 'yes' features)
binary_cols = ['mainroad', 'guestroom', 'basement', 'hotwaterheating', 'airconditioning', 'prefarea']
df['amenities_score'] = df[binary_cols].apply(lambda x: (x == 'yes').sum(), axis=1)

KeyError: "None of [Index(['mainroad', 'guestroom', 'basement', 'hotwaterheating',\n       'airconditioning', 'prefarea'],\n      dtype='str')] are in the [columns]"

In [5]:
# 2. Area per Bedroom (Space efficiency)
df['area_per_bedroom'] = df['area'] / df['bedrooms']

In [6]:
# 3. Log Transformation of Price (Handling skewness identified in EDA)
df['price_log'] = np.log1p(df['price'])

### Type of Features

**Numeric Features**

In [7]:
num_features = [feature for feature in df.columns if df[feature].dtype != 'O']
print('Num of Numerical Features :', len(num_features))

Num of Numerical Features : 16


**Categorical Features**

In [8]:
cat_features = [feature for feature in df.columns if df[feature].dtype == 'O']
print('Num of Categorical Features :', len(cat_features))

Num of Categorical Features : 0


**Discrete features**

In [9]:
discrete_features=[feature for feature in num_features if len(df[feature].unique())<=25]
print('Num of Discrete Features :',len(discrete_features))

Num of Discrete Features : 12


**Continues Features**

In [10]:
continuous_features=[feature for feature in num_features if feature not in discrete_features]
print('Num of Continuous Features :',len(continuous_features))

Num of Continuous Features : 4


### Train-Test Split & Pipeline Construction

In [11]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

In [12]:
# Independent Features (The 'Questions')
# Modern way: Remove 'axis=1'
X = df.drop(columns=['price', 'price_log'])

# Dependent Feature (The 'Answer')
y = df['price_log']

In [13]:
X_train, X_test, y_train, y_test = train_test_split(
    X,y, test_size=0.2, random_state=42
)

In [14]:
print(f"Training set size: {X_train.shape[0]} houses")
print(f"Testing set size: {X_test.shape[0]} houses")

Training set size: 436 houses
Testing set size: 109 houses


In [15]:
# Define columns
num_features = ['area', 'bedrooms', 'bathrooms', 'stories', 'parking', 'amenities_score', 'area_per_bedroom']
cat_features = ['mainroad', 'guestroom', 'basement', 'hotwaterheating', 'airconditioning', 'prefarea', 'furnishingstatus']

In [16]:
# Preprocessing Pipeline
preprocessor = ColumnTransformer([
    ("NumPipeline", StandardScaler(), num_features),
    ("CatPipeline", OneHotEncoder(drop='first'), cat_features)
])

###  Model Comparison & Hyperparameter Tuning

In [17]:
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, AdaBoostRegressor
from catboost import CatBoostRegressor
from sklearn.metrics import r2_score, mean_absolute_error

In [18]:
models = {
    "Linear Regression": LinearRegression(),
    "Lasso": Lasso(),
    "Ridge": Ridge(),
    "Random Forest": RandomForestRegressor(),
    "CatBoost": CatBoostRegressor(verbose=False),
    "AdaBoost": AdaBoostRegressor()
}

In [19]:
from sklearn.metrics import r2_score

def evaluate_models(X_train, y_train, X_test, y_test, models):
    report = {}
    for name, model in models.items():
        model.fit(X_train, y_train)
        
        # Scoring both sets to check for overfitting
        train_score = r2_score(y_train, model.predict(X_train))
        test_score = r2_score(y_test, model.predict(X_test))
        
        report[name] = {
            "Train R2": train_score,
            "Test R2": test_score
        }
    return report

### Final Model Export

In [20]:
import dill
import os
from sklearn.pipeline import Pipeline

# Assuming results_df and models are available from previous executed cells
# Find the best model name based on R2 Score from results_df
best_model_name = results_df.loc[results_df['R2 Score'].idxmax()]['Model']

# Retrieve the best model object from the 'models' dictionary
best_model = models[best_model_name]

# Create final pipeline using the preprocessor (defined in cell 6-eb9gfqUebz)
final_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", best_model)
])

# Ensure artifacts directory exists
os.makedirs('artifacts', exist_ok=True)

# Save the final pipeline as an artifact
with open('artifacts/model_pipeline.pkl', 'wb') as f:
    dill.dump(final_pipeline, f)

print(f"Successfully saved the final pipeline with the best model: {best_model_name}")

NameError: name 'results_df' is not defined

In [ ]:
display(results_df)

,Model,MSE,MAE,RMSE,R2 Score
3,Random Forest,0.061714,0.190762,0.248423,0.680386
0,Linear Regression,0.063288,0.199895,0.251571,0.672233
2,Ridge,0.063332,0.199881,0.251658,0.672007
4,CatBoost,0.065380,0.198901,0.255694,0.661402
5,AdaBoost,0.074722,0.219840,0.273354,0.613015
1,Lasso,0.193964,0.364673,0.440413,-0.004531


### Predictions

In [ ]:
import dill

# Load the final_pipeline which includes the preprocessor and the best model
with open('artifacts/model_pipeline.pkl', 'rb') as f:
    final_pipeline = dill.load(f)

y_train_pred = final_pipeline.predict(X_train)
y_test_pred = final_pipeline.predict(X_test)


In [ ]:
from sklearn.metrics import mean_squared_error

def evaluate(y_true, y_pred, name="Dataset"):
    r2 = r2_score(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))

    print(f"\n{name} Performance")
    print(f"R2 Score: {r2:.4f}")
    print(f"MAE     : {mae:.2f}")
    print(f"RMSE    : {rmse:.2f}")


evaluate(y_train, y_train_pred, "Training")
evaluate(y_test, y_test_pred, "Test")


Training Performance
R2 Score: 0.9429
MAE     : 0.06
RMSE    : 0.08

Test Performance
R2 Score: 0.6733
MAE     : 0.20
RMSE    : 0.25
